# 实现一个动物专家系统

来自 [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) 的示例。

在这个示例中，我们将实现一个简单的基于知识的系统，通过一些物理特征来判断动物。该系统可以用以下的 AND-OR 树表示（这是整个树的一部分，我们可以轻松添加更多规则）：

![](../../../../lessons/2-Symbolic/images/AND-OR-Tree.png)


## 我们自己的基于反向推理的专家系统外壳

让我们尝试基于产生式规则定义一种简单的知识表示语言。我们将使用 Python 类作为关键字来定义规则。主要会有三种类型的类：
* `Ask` 表示需要向用户提问的问题。它包含一组可能的答案。
* `If` 表示一条规则，它只是用于存储规则内容的语法糖。
* `AND`/`OR` 是用于表示树的 AND/OR 分支的类。它们仅存储内部的参数列表。为了简化代码，所有功能都定义在父类 `Content` 中。


In [ ]:
class Ask():
    def __init__(self,choices=['y','n']):
        self.choices = choices
    def ask(self):
        if max([len(x) for x in self.choices])>1:#排除默认的y,n选项
            for i,x in enumerate(self.choices):
                print("{0}. {1}".format(i,x),flush=True)
            x = int(input())#操，这里又有个x
            return self.choices[x]
        else:
            print("/".join(self.choices),flush=True)
            return input()

class Content():
    def __init__(self,x):
        self.x=x
        
class If(Content):
    pass

class AND(Content):
    pass

class OR(Content):
    pass

在我们的系统中，工作记忆将包含作为**属性-值对**的**事实**列表。知识库可以定义为一个大的字典，将动作（应插入工作记忆的新事实）映射到条件，这些条件以AND-OR表达式表示。此外，一些事实可以被`询问`。


In [2]:
rules = {
    'default': Ask(['y','n']),
    'color' : Ask(['red-brown','black and white','other']),
    'pattern' : Ask(['dark stripes','dark spots']),
    'mammal': If(OR(['hair','gives milk'])),
    'carnivor': If(OR([AND(['sharp teeth','claws','forward-looking eyes']),'eats meat'])),
    'ungulate': If(['mammal',OR(['has hooves','chews cud'])]),
    'bird': If(OR(['feathers',AND(['flies','lies eggs'])])),
    'animal:monkey' : If(['mammal','carnivor','color:red-brown','pattern:dark spots']),
    'animal:tiger' : If(['mammal','carnivor','color:red-brown','pattern:dark stripes']),
    'animal:giraffe' : If(['ungulate','long neck','long legs','pattern:dark spots']),
    'animal:zebra' : If(['ungulate','pattern:dark stripes']),
    'animal:ostrich' : If(['bird','long nech','color:black and white','cannot fly']),
    'animal:pinguin' : If(['bird','swims','color:black and white','cannot fly']),
    'animal:albatross' : If(['bird','flies well'])
}

为了进行反向推理，我们将定义一个 `Knowledgebase` 类。它将包含以下内容：
* 工作的 `memory` - 一个将属性映射到值的字典
* 知识库的 `rules`，格式如上所述

两个主要方法是：
* `get` 用于获取某个属性的值，如果需要会执行推理。例如，`get('color')` 将获取颜色槽的值（如果需要会询问，并将值存储在工作内存中以供后续使用）。如果我们询问 `get('color:blue')`，它会询问颜色，然后根据颜色返回 `y`/`n` 值。
* `eval` 执行实际的推理，即遍历 AND/OR 树，评估子目标等。


In [3]:
class KnowledgeBase():
    def __init__(self,rules):
        self.rules = rules
        self.memory = {}
        
    def get(self,name):
        if ':' in name:
            k,v = name.split(':')
            vv = self.get(k)
            return 'y' if v==vv else 'n'
        if name in self.memory.keys():
            return self.memory[name]
        for fld in self.rules.keys():
            if fld==name or fld.startswith(name+":"):
                # print(" + proving {}".format(fld))
                value = 'y' if fld==name else fld.split(':')[1]#else对应的是fld.startswith的情况
                res = self.eval(self.rules[fld],field=name)
                if res!='y' and res!='n' and value=='y':#res的值可能是y/n/或具体的值如red,black and white
                    self.memory[name] = res
                    return res
                if res=='y':
                    self.memory[name] = value
                    return value
        # field is not found, using default
        res = self.eval(self.rules['default'],field=name)
        self.memory[name]=res
        return res
                
    def eval(self,expr,field=None):
        # print(" + eval {}".format(expr))
        if isinstance(expr,Ask):#判断是什么类型
            print(field)
            return expr.ask()
        elif isinstance(expr,If):
            return self.eval(expr.x)
        elif isinstance(expr,AND) or isinstance(expr,list):
            expr = expr.x if isinstance(expr,AND) else expr
            for x in expr:
                if self.eval(x)=='n':
                    return 'n'
            return 'y'
        elif isinstance(expr,OR):
            for x in expr.x:
                if self.eval(x)=='y':
                    return 'y'
            return 'n'
        elif isinstance(expr,str):
            return self.get(expr)#是字符串就询问是不是正确的，会用default y，n判断
        else:
            print("Unknown expr: {}".format(expr))

现在让我们定义我们的动物知识库并进行咨询。请注意，此操作会向您提问。您可以通过输入 `y`/`n` 来回答是非问题，或者通过指定数字（0..N）来回答有多个选项的问题。


In [5]:
kb = KnowledgeBase(rules)
kb.get('animal')

hair
y/n
sharp teeth
y/n
eats meat
y/n
carnivor
y/n
color
0. red-brown
1. black and white
2. other
has hooves
y/n
chews cud
y/n
ungulate
y/n
feathers
y/n
long nech
y/n
swims
y/n
flies well
y/n


'albatross'

## 使用 PyKnow 进行前向推理

在下一个示例中，我们将尝试使用一个知识表示库 [PyKnow](https://github.com/buguroo/pyknow/) 来实现前向推理。**PyKnow** 是一个用于在 Python 中创建前向推理系统的库，其设计类似于经典的旧系统 [CLIPS](http://www.clipsrules.net/index.html)。

我们本可以自己实现前向链推理，且不会遇到太多问题，但简单的实现通常效率不高。为了更高效地进行规则匹配，使用了一种特殊的算法 [Rete](https://en.wikipedia.org/wiki/Rete_algorithm)。


In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/buguroo/pyknow/

  Cloning https://github.com/buguroo/pyknow/ to c:\users\huawei\appdata\local\temp\pip-req-build-3vb2oydc


  Running command git clone --filter=blob:none --quiet https://github.com/buguroo/pyknow/ 'C:\Users\huawei\AppData\Local\Temp\pip-req-build-3vb2oydc'
  fatal: unable to access 'https://github.com/buguroo/pyknow/': Recv failure: Connection was reset
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/buguroo/pyknow/ 'C:\Users\huawei\AppData\Local\Temp\pip-req-build-3vb2oydc' did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/buguroo/pyknow/ 'C:\Users\huawei\AppData\Local\Temp\pip-req-build-3vb2oydc' did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
from pyknow import *
#import pyknow

我们将把我们的系统定义为一个继承自 `KnowledgeEngine` 的类。每条规则由一个带有 `@Rule` 注解的单独函数定义，该注解指定规则何时触发。在规则内部，我们可以使用 `declare` 函数添加新事实，添加这些事实将导致前向推理引擎调用更多规则。


In [14]:
class Animals(KnowledgeEngine):
    @Rule(OR(
           AND(Fact('sharp teeth'),Fact('claws'),Fact('forward looking eyes')),
           Fact('eats meat')))
    def cornivor(self):
        self.declare(Fact('carnivor'))
        
    @Rule(OR(Fact('hair'),Fact('gives milk')))
    def mammal(self):
        self.declare(Fact('mammal'))

    @Rule(Fact('mammal'),
          OR(Fact('has hooves'),Fact('chews cud')))
    def hooves(self):
        self.declare('ungulate')
        
    @Rule(OR(Fact('feathers'),AND(Fact('flies'),Fact('lays eggs'))))
    def bird(self):
        self.declare('bird')
        
    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark spots'))
    def monkey(self):
        self.declare(Fact(animal='monkey'))

    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark stripes'))
    def tiger(self):
        self.declare(Fact(animal='tiger'))

    @Rule(Fact('ungulate'),
          Fact('long neck'),
          Fact('long legs'),
          Fact(pattern='dark spots'))
    def giraffe(self):
        self.declare(Fact(animal='giraffe'))

    @Rule(Fact('ungulate'),
          Fact(pattern='dark stripes'))
    def zebra(self):
        self.declare(Fact(animal='zebra'))

    @Rule(Fact('bird'),
          Fact('long neck'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def straus(self):
        self.declare(Fact(animal='ostrich'))

    @Rule(Fact('bird'),
          Fact('swims'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def pinguin(self):
        self.declare(Fact(animal='pinguin'))

    @Rule(Fact('bird'),
          Fact('flies well'))
    def albatros(self):
        self.declare(Fact(animal='albatross'))
        
    @Rule(Fact(animal=MATCH.a))
    def print_result(self,a):
          print('Animal is {}'.format(a))
                    
    def factz(self,l):
        for x in l:
            self.declare(x)

一旦我们定义了一个知识库，我们会用一些初始事实填充工作内存，然后调用`run()`方法来执行推理。结果可以看到新的推导事实被添加到工作内存中，包括关于动物的最终事实（如果我们正确设置了所有初始事实）。


In [15]:
ex1 = Animals()
ex1.reset()
ex1.factz([
    Fact(color='red-brown'),
    Fact(pattern='dark stripes'),
    Fact('sharp teeth'),
    Fact('claws'),
    Fact('forward looking eyes'),
    Fact('gives milk')])
ex1.run()
ex1.facts

Animal is tiger


FactList([(0, InitialFact()),
          (1, Fact(color='red-brown')),
          (2, Fact(pattern='dark stripes')),
          (3, Fact('sharp teeth')),
          (4, Fact('claws')),
          (5, Fact('forward looking eyes')),
          (6, Fact('gives milk')),
          (7, Fact('mammal')),
          (8, Fact('carnivor')),
          (9, Fact(animal='tiger'))])


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
